# YOLO26m — Continue Training +50 Epochs from Current Best

This notebook starts from:

```text
/content/drive/MyDrive/inferance/best.pt
```

and performs **50 additional low-learning-rate fine-tuning epochs** on the exact same train/validation split.

The original checkpoint is never overwritten.

Output:

```text
MyDrive/inferance/YOLO26m_Extended_100ep/
```

The held-out detector test split is not used.


In [ ]:
# CELL 1 — INSTALL + DRIVE + GPU

!pip -q install ultralytics==8.4.117

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import defaultdict
import json, shutil, random, time, gc

import numpy as np
import torch
from ultralytics import YOLO
import ultralytics

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU runtime in Colab.')

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)

if VRAM_GB >= 38:
    BATCH = 12
elif VRAM_GB >= 22:
    BATCH = 8
elif VRAM_GB >= 14:
    BATCH = 4
else:
    BATCH = 2

print('GPU:', GPU_NAME)
print(f'VRAM: {VRAM_GB:.1f} GB')
print('Torch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('Batch:', BATCH)
print('STATUS: ENVIRONMENT_READY')


In [ ]:
# CELL 2 — PATHS

MYDRIVE = Path('/content/drive/MyDrive')

PROJECT_ROOT = MYDRIVE / 'Coral_Reef_AI_Project'
DET_ROOT = PROJECT_ROOT / '01_Data' / 'Final' / 'Detection_v1'

TRAIN_IMG = DET_ROOT / 'images' / 'train'
VAL_IMG = DET_ROOT / 'images' / 'validation'

TRAIN_COCO = DET_ROOT / 'annotations' / 'instances_train.json'
VAL_COCO = DET_ROOT / 'annotations' / 'instances_validation.json'

OLD_BEST = MYDRIVE / 'inferance' / 'best.pt'

EXT_ROOT = MYDRIVE / 'inferance' / 'YOLO26m_Extended_100ep'
EXT_ROOT.mkdir(parents=True, exist_ok=True)

for p in [TRAIN_IMG, VAL_IMG, TRAIN_COCO, VAL_COCO, OLD_BEST]:
    if not p.exists():
        raise FileNotFoundError(p)

print('Old best:', OLD_BEST)
print('Output:', EXT_ROOT)
print('STATUS: PATHS_READY')


In [ ]:
# CELL 3 — REBUILD SAME YOLO DATASET ADAPTER

YOLO_DATA = Path('/content/Coral_YOLO26_Extended_Dataset')

if YOLO_DATA.exists():
    shutil.rmtree(YOLO_DATA)

for split in ['train', 'validation']:
    (YOLO_DATA / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA / 'labels' / split).mkdir(parents=True, exist_ok=True)

def resolve_source(folder, file_name):
    name = Path(file_name).name
    p = folder / name
    if p.exists():
        return p
    matches = list(folder.glob(Path(name).stem + '.*'))
    if not matches:
        raise FileNotFoundError(name)
    return matches[0]

def prep_yolo(coco_path, src_dir, split):
    data = json.loads(coco_path.read_text(encoding='utf-8'))
    by_img = defaultdict(list)

    for a in data['annotations']:
        by_img[int(a['image_id'])].append(a)

    ni = 0
    nbx = 0

    for im in data['images']:
        src = resolve_source(src_dir, im['file_name'])
        dst = YOLO_DATA / 'images' / split / src.name
        shutil.copy2(src, dst)

        W = float(im['width'])
        H = float(im['height'])
        lines = []

        for a in by_img.get(int(im['id']), []):
            x, y, w, h = map(float, a['bbox'])
            xc = (x + w/2) / W
            yc = (y + h/2) / H
            lines.append(f'0 {xc:.8f} {yc:.8f} {w/W:.8f} {h/H:.8f}')
            nbx += 1

        (YOLO_DATA / 'labels' / split / f'{dst.stem}.txt').write_text(
            '\n'.join(lines),
            encoding='utf-8'
        )
        ni += 1

    return ni, nbx

tr_i, tr_b = prep_yolo(TRAIN_COCO, TRAIN_IMG, 'train')
va_i, va_b = prep_yolo(VAL_COCO, VAL_IMG, 'validation')

assert (tr_i, tr_b, va_i, va_b) == (452, 1540, 100, 354)

DATA_YAML = YOLO_DATA / 'data.yaml'
DATA_YAML.write_text(
    'path: ' + str(YOLO_DATA) + '\n'
    'train: images/train\n'
    'val: images/validation\n'
    'names:\n'
    '  0: target_coral\n',
    encoding='utf-8'
)

print('Train:', tr_i, 'images /', tr_b, 'boxes')
print('Val:', va_i, 'images /', va_b, 'boxes')
print('YAML:', DATA_YAML)
print('STATUS: DATA_READY')


In [ ]:
# CELL 4 — VALIDATE CURRENT best.pt

old_model = YOLO(str(OLD_BEST))

old_val = old_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=960,
    batch=BATCH,
    conf=0.001,
    iou=0.70,
    max_det=500,
    device=0,
    workers=2,
    plots=False,
    verbose=True
)

OLD_METRICS = {
    'mAP50_95': float(old_val.box.map),
    'mAP50': float(old_val.box.map50),
    'mAP75': float(old_val.box.map75),
    'precision': float(getattr(old_val.box, 'mp', np.nan)),
    'recall': float(getattr(old_val.box, 'mr', np.nan))
}

print('CURRENT BEST VALIDATION')
for k, v in OLD_METRICS.items():
    print(k, ':', v)

(EXT_ROOT / 'old_best_validation.json').write_text(
    json.dumps(OLD_METRICS, indent=2),
    encoding='utf-8'
)

del old_model, old_val
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# CELL 5 — +50 EPOCH CONTINUATION FROM CURRENT BEST

EXTRA_EPOCHS = 50
IMGSZ = 960
LR0 = 5e-5
LRF = 0.10
RUN_NAME = 'FineTune_From_Best_Plus50'

model = YOLO(str(OLD_BEST))

print('Start checkpoint:', OLD_BEST)
print('Additional epochs:', EXTRA_EPOCHS)
print('imgsz:', IMGSZ)
print('batch:', BATCH)
print('lr0:', LR0)

t0 = time.time()

model.train(
    data=str(DATA_YAML),
    epochs=EXTRA_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,

    device=0,
    workers=2,

    optimizer='AdamW',
    lr0=LR0,
    lrf=LRF,
    weight_decay=5e-4,

    cos_lr=True,
    warmup_epochs=0.5,
    patience=20,

    save=True,
    save_period=5,

    seed=SEED,
    deterministic=True,

    degrees=2.0,
    translate=0.04,
    scale=0.10,
    shear=0.0,
    perspective=0.0,

    fliplr=0.50,
    flipud=0.05,

    hsv_h=0.010,
    hsv_s=0.15,
    hsv_v=0.12,

    mosaic=0.05,
    mixup=0.0,
    close_mosaic=10,

    multi_scale=0.05,
    max_det=500,

    amp=True,

    project=str(EXT_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=True
)

RUN_DIR = EXT_ROOT / RUN_NAME
NEW_BEST = RUN_DIR / 'weights' / 'best.pt'
NEW_LAST = RUN_DIR / 'weights' / 'last.pt'

if not NEW_BEST.exists():
    raise FileNotFoundError(NEW_BEST)

print(f'Training minutes: {(time.time()-t0)/60:.2f}')
print('NEW BEST:', NEW_BEST)
print('NEW LAST:', NEW_LAST)
print('STATUS: EXTRA_50_COMPLETE')


In [ ]:
# CELL 6 — VALIDATE NEW BEST + COMPARE

gc.collect()
torch.cuda.empty_cache()

new_model = YOLO(str(NEW_BEST))

new_val = new_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=960,
    batch=BATCH,
    conf=0.001,
    iou=0.70,
    max_det=500,
    device=0,
    workers=2,
    plots=True,
    verbose=True
)

NEW_METRICS = {
    'mAP50_95': float(new_val.box.map),
    'mAP50': float(new_val.box.map50),
    'mAP75': float(new_val.box.map75),
    'precision': float(getattr(new_val.box, 'mp', np.nan)),
    'recall': float(getattr(new_val.box, 'mr', np.nan))
}

comparison = {
    k: {
        'old': OLD_METRICS[k],
        'new': NEW_METRICS[k],
        'delta': NEW_METRICS[k] - OLD_METRICS[k]
    }
    for k in OLD_METRICS
}

print('='*100)
print('OLD vs EXTENDED')
print('='*100)
for k, d in comparison.items():
    print(
        f"{k:12s} | old={d['old']:.6f}"
        f" | new={d['new']:.6f}"
        f" | delta={d['delta']:+.6f}"
    )

(EXT_ROOT / 'new_best_validation.json').write_text(
    json.dumps(NEW_METRICS, indent=2),
    encoding='utf-8'
)

(EXT_ROOT / 'old_vs_new_comparison.json').write_text(
    json.dumps(comparison, indent=2),
    encoding='utf-8'
)

print('NEW BEST:', NEW_BEST)
print('STATUS: COMPARISON_COMPLETE')


In [ ]:
# CELL 7 — PROMOTE ONLY IF mAP50:95 IMPROVED

PROMOTED = MYDRIVE / 'inferance' / 'best_yolo26m_extended.pt'

improved = NEW_METRICS['mAP50_95'] > OLD_METRICS['mAP50_95']

print('mAP50:95 improved:', improved)

if improved:
    shutil.copy2(NEW_BEST, PROMOTED)
    print('PROMOTED:', PROMOTED)
else:
    print('Old best remains better on mAP50:95.')
    print('New checkpoint kept here:', NEW_BEST)

print('Old best remains untouched:', OLD_BEST)
print('STATUS: DONE')


## Next step

If `best_yolo26m_extended.pt` is created, rerun the detector confidence sweep on validation before using it in end-to-end/video inference.

Do not touch the detector held-out test yet.
